In [ ]:
#| default_exp preprocessing.zero_degree_solder_pin.fit_rotated_rectangles_to_masks

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#| export
import sys
from pathlib import Path
from fastcore.script import *

In [ ]:
#| export
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(CV_TOOLS))

In [ ]:
#| export
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
sys.path.append(str(custom_lib_path))

In [ ]:
#| export
from cv_tools.imports import *
from cv_tools.core import *
from dotenv import load_dotenv
from collections import Counter


In [ ]:
#| export
load_dotenv(dotenv_path=f'/home/ai_sintercra/homes/hasan/projects/git_data/new_test/new_test/.env')

False

In [ ]:
#| exporti
from new_test.core import *

In [ ]:
from platform import system
if system() == 'Windows':
    core_path = Path(r'E:\CurrentTrainingData20240812_trn_val/trn_masks')
else:
    core_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/trn_masks')
#sys.path.append(str(core_path))

#| export

## fit rotated rectangle to the mask (correct masks)

In [ ]:
MODEL_NAME='ETPD-WAR1_03.keras'
mask_path = Path(
    core_path.parent, 
    f'Incoming_1B_Loetstift/Incoming_1B_Loetstift_unzip/main_im2_cropped_masks/{MODEL_NAME}/passed/masks'
    )

In [ ]:
#| export
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [ ]:
#| export
def convert_to_rotated_rect(
    mask: np.ndarray,
    )->np.ndarray:
    """Convert any mask shape to a rotated rectangle mask.
    
    Args:
        mask: Binary mask array
        
    Returns:
        Binary mask array with only the rotated rectangle
    """
    # Find contours
    contours, _ = cv2.findContours(
        mask.astype(np.uint8), 
        cv2.RETR_EXTERNAL, 
        cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return mask
    
    # Create empty mask
    rect_mask = np.zeros_like(mask)
    
    # Process each contour
    for contour in contours:
        # Get rotated rectangle
        rect = cv2.minAreaRect(contour)
        box = cv2.boxPoints(rect)
        box = np.int64(box)
        
        # Draw filled rotated rectangle
        cv2.drawContours(rect_mask, [box], 0, 255, -1)
    
    return rect_mask

In [ ]:
nrm_msk = read_img(mask_path.ls()[0])
rect_msk = convert_to_rotated_rect(nrm_msk)

In [ ]:
#| export
def process_masks_parallel(mask_paths, n_workers=None):
    """Process multiple masks in parallel to convert them to rotated rectangles.
    
    Args:
        mask_paths: List of Path objects to mask files
        n_workers: Number of parallel workers. If None, uses all available CPUs
        
    Returns:
        List of processed mask arrays
    """
    if n_workers is None:
        n_workers = os.cpu_count()
    
    def process_single(path):
        mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            return None
        return convert_to_rotated_rect(mask)
    
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = [executor.submit(process_single, path) for path in mask_paths]
        results = []
        for future in tqdm(futures, total=len(mask_paths), desc="Converting masks to rotated rectangles"):
            results.append(future.result())
    
    return [r for r in results if r is not None]

In [ ]:
#| export
def save_processed_masks(mask_paths, output_dir, n_workers=None):
    """Process and save converted masks in parallel.
    
    Args:
        mask_paths: List of Path objects to mask files
        output_dir: Directory to save processed masks
        n_workers: Number of parallel workers
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    processed_masks = []
    for path in tqdm(mask_paths, desc="Converting masks to rotated rectangles"):
        mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            processed_mask = convert_to_rotated_rect(mask)
            processed_masks.append(processed_mask)
    
    for path, mask in zip(mask_paths, processed_masks):
        if mask is not None:
            output_path = output_dir / path.name
            cv2.imwrite(str(output_path), mask)

In [ ]:
#| export
@call_parse
def main_(
    mask_path:Param("Path to the mask file", Path)='name/of/mask/path',
    save_path:Param("Path to save the processed mask", Path)='name/of/save/path',
    n_workers:Param("Number of parallel workers", int)=None,
    )->None:
    """Main function to process and save rotated rectangle masks.
    
    Args:
        mask_path: Path to the mask file
        save_path: Path to save the processed mask
    """
    masks = mask_path.ls()
    save_processed_masks(
        mask_paths=masks,
        output_dir=save_path,
        n_workers=n_workers
    )

In [ ]:
fns = core_path.ls()

In [ ]:
fns[0].name

'src_img_recipe_105_idx_3_gen_image_0_image_1.png'

In [ ]:
np.vectorize(get_pin_type)

In [ ]:
#| export
def pin_type(x):
    return get_pin_type()[get_m_name(str(x))]

In [ ]:
pin_type_vec = np.vectorize(pin_type)	

In [ ]:
name2rec = {Path(i).name: get_m_name(str(i)) for i in fns}
name2pintype = {Path(i).name: get_pin_type()[get_m_name(str(i))] for i in fns}

In [ ]:
solder_pins = []
for k,v in name2pintype.items():
    if ('solder' in v):
        if 'gen' not in v:
            solder_pins.append(k)

In [ ]:
# filer for solder pins
solder_pins = {k: v for k, v in name2pintype.items() if (v != 'undefined_or_generated') and ('solder' in v) and 'gen_image' not in k}

In [ ]:
len(solder_pins)

2710

In [ ]:
core_path.parent

Path('E:/CurrentTrainingData20240812_trn_val')

In [ ]:
solder_images = Path(core_path.parent, 'solder_pins', 'images')
solder_masks = Path(core_path.parent, 'solder_pins', 'masks')
solder_metadata = Path(core_path.parent, 'solder_pins', 'metadata')
dir_solder_pins = [

    solder_images,
    solder_masks,
    solder_metadata
]
for i in dir_solder_pins:
    Path(i).mkdir(parents=True, exist_ok=True)


In [ ]:
#| export
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import shutil
from tqdm.auto import tqdm

def copy_file(src_dest_pair):
    src, dest = src_dest_pair
    shutil.copy2(src, dest)

In [ ]:
#| export
def create_copy_pairs(
    solder_pin_fns, 
    src_img_path, 
    src_msk_path, 
    dst_img_path, 
    dst_msk_path):
    """Create pairs of source and destination files for copying solder pin images and masks.
    
    Args:
        solder_pin_fns (list): List of solder pin image names
        src_img_path (Path): Base path containing source images
        src_mask_path (Path): Base path containing source masks
        dst_img_path (Path): Destination path for solder pin images
        dst_mask_path (Path): Destination path for solder pin masks
        
    Returns:
        list: List of tuples containing (source_path, destination_path) pairs
    """
    copy_pairs = []
    for img_name in solder_pin_fns:
        src_img = Path(src_img_path, img_name)
        src_msk = Path(src_msk_path, img_name)
        
        dst_img_name = Path(dst_img_path, img_name)
        dst_msk_name = Path(dst_msk_path, img_name)
        
        copy_pairs.extend([
            (src_img, dst_img_name),
            (src_msk, dst_msk_name)
        ])
    return copy_pairs

In [ ]:
core_path

Path('E:/CurrentTrainingData20240812_trn_val/trn_masks')

In [ ]:
solder_pin_fns = list(solder_pins.keys())
src_img_path = Path(core_path.parent,'trn_images')
src_msk_path = Path(core_path.parent,'trn_masks')
dst_img_path = Path(core_path.parent,'solder_pins','images')
dst_msk_path = Path(core_path.parent,'solder_pins','masks')

In [ ]:

# Create pairs using the function
copy_pairs = create_copy_pairs(
    solder_pin_fns=solder_pin_fns, 
    src_img_path=src_img_path, 
    src_msk_path=src_msk_path, 
    dst_img_path=dst_img_path, 
    dst_msk_path=dst_msk_path)

In [ ]:
copy_pairs

[(Path('E:/CurrentTrainingData20240812_trn_val/trn_images/74_In_18.png'),
  Path('E:/CurrentTrainingData20240812_trn_val/solder_pins/images/74_In_18.png')),
 (Path('E:/CurrentTrainingData20240812_trn_val/trn_masks/74_In_18.png'),
  Path('E:/CurrentTrainingData20240812_trn_val/solder_pins/masks/74_In_18.png')),
 (Path('E:/CurrentTrainingData20240812_trn_val/trn_images/18_In_18.png'),
  Path('E:/CurrentTrainingData20240812_trn_val/solder_pins/images/18_In_18.png')),
 (Path('E:/CurrentTrainingData20240812_trn_val/trn_masks/18_In_18.png'),
  Path('E:/CurrentTrainingData20240812_trn_val/solder_pins/masks/18_In_18.png')),
 (Path('E:/CurrentTrainingData20240812_trn_val/trn_images/190_In_18.png'),
  Path('E:/CurrentTrainingData20240812_trn_val/solder_pins/images/190_In_18.png')),
 (Path('E:/CurrentTrainingData20240812_trn_val/trn_masks/190_In_18.png'),
  Path('E:/CurrentTrainingData20240812_trn_val/solder_pins/masks/190_In_18.png')),
 (Path('E:/CurrentTrainingData20240812_trn_val/trn_images/19

In [ ]:
#| export
def parallel_copy_files(copy_pairs, max_workers=8):
    """Copy files in parallel using ThreadPoolExecutor.
    
    Args:
        copy_pairs (list): List of (source, destination) path pairs to copy
        max_workers (int): Maximum number of worker threads to use
        
    Returns:
        None
    """
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(tqdm(
            executor.map(copy_file, copy_pairs),
            total=len(copy_pairs),
            desc="Copying solder pin files"
        ))


Copying solder pin files:  54%|█████▍    | 2934/5420 [1:00:55<51:37,  1.25s/it]  


In [ ]:

# Use the function
parallel_copy_files(copy_pairs)

In [ ]:
Counter(name2pintype.values())
#| export

Counter({'pressfit1B': 7229,
         'pressfit2B': 3542,
         'undefined_or_generated': 3457,
         'solder1B': 1998,
         'solder2B': 1411,
         'OneType': 142,
         'pressfitSMART': 115})

In [ ]:
#| export
CURRETNT_NB='/home/ai_sintercra/homes/hasan/projects/git_data/new_test/nbs'

In [ ]:
#| export
def find_rotated_rectangles(mask_path, angle_threshold=5):
    """Find rotated rectangles in a binary mask image.
    
    Args:
        mask_path (Path): Path to binary mask image
        angle_threshold (float): Minimum angle in degrees to consider a rectangle as rotated
        
    Returns:
        bool: True if image contains rotated rectangles, False otherwise
    """
    # Read mask image
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return False
        
    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Check each contour for rotated rectangles
    for contour in contours:
        # Get minimum area rectangle
        rect = cv2.minAreaRect(contour)
        box = cv2.boxPoints(rect)
        box = np.int0(box)
        
        # Get angle
        angle = rect[-1]
        # OpenCV's minAreaRect returns angle in range [-90, 0) degrees
        # When angle is less than -45, it means the rectangle is rotated more than 45 degrees
        # Adding 90 normalizes the angle to be in range [0, 90) degrees
        if angle < -45:
            angle += 90
            
        # If angle exceeds threshold, consider it rotated
        # Check if angle exceeds threshold to consider rectangle as rotated
        if abs(angle) > angle_threshold:
            return True
            
    return False

In [ ]:
#| export
def copy_rotated_rectangles(src_img_dir, src_msk_dir, dst_img_dir, dst_msk_dir, angle_threshold=5):
    """Copy images and masks containing rotated rectangles to separate folders.
    
    Args:
        src_img_dir (Path): Source images directory
        src_msk_dir (Path): Source masks directory
        dst_img_dir (Path): Destination images directory
        dst_msk_dir (Path): Destination masks directory
        angle_threshold (float): Minimum angle in degrees to consider a rectangle as rotated
    """
    # Create destination directories if they don't exist
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_msk_dir.mkdir(parents=True, exist_ok=True)
    
    # Get all mask files
    mask_files = list(src_msk_dir.glob('*.png'))
    
    # Find masks with rotated rectangles
    rotated_masks = []
    for mask_path in tqdm(mask_files, desc="Finding rotated rectangles"):
        if find_rotated_rectangles(mask_path, angle_threshold):
            rotated_masks.append(mask_path)
            
    # Create copy pairs for rotated rectangles
    copy_pairs = []
    for mask_path in rotated_masks:
        img_path = src_img_dir / mask_path.name
        dst_img_path = dst_img_dir / mask_path.name
        dst_msk_path = dst_msk_dir / mask_path.name
        
        if img_path.exists():
            copy_pairs.extend([
                (img_path, dst_img_path),
                (mask_path, dst_msk_path)
            ])
            
    # Copy files in parallel
    parallel_copy_files(copy_pairs)
    
    print(f"Found {len(rotated_masks)} images with rotated rectangles")
    print(f"Copied {len(copy_pairs)//2} image-mask pairs to destination folders")

# Example usage:
# copy_rotated_rectangles(
#     src_img_dir=Path('path/to/source/images'),
#     src_msk_dir=Path('path/to/source/masks'),
#     dst_img_dir=Path('path/to/destination/images'),
#     dst_msk_dir=Path('path/to/destination/masks')
# )


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('39_preprocessing.zero_degree_solder_pin.ipynb')

ValueError: '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\git_data\\nbs\\26_model_evaluation.new_model.ipynb' is not in the subpath of '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\git_data\\new_test\\nbs'